# Base Qwen vs biological-SFT checkpoint vs TxGemma

This notebook follows `01_base_model_selection_pilot.ipynb` and `03b_analyze_bio_knowledge_sft.ipynb`. It compares four frozen variants on the same canonical development data:

1. the untouched Qwen base used for SFT;
2. the user-selected biological LoRA checkpoint attached to that same base; and
3. `google/txgemma-9b-chat`; and
4. `google/txgemma-27b-chat`.

The primary comparison is `base_selection.jsonl`, the generated, verifiable portion of canonical dev used by the original pilot. A deterministic LAB-Bench slice from `dev.jsonl` is reported separately. Models are loaded one at a time and all results stay in memory—this notebook writes no metrics, predictions, plots, or manifests.


## Setup and authentication

Install a recent `transformers`, `peft`, `accelerate`, and (for 4-bit loading) `bitsandbytes` in the GPU runtime. TxGemma access may require accepting Google's model terms on Hugging Face. Set `HF_TOKEN` in the environment or allow `notebook_login()` to prompt when model execution is enabled.


In [ ]:
from __future__ import annotations

import gc
import math
import os
import random
import sys
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'training_utils.py').is_file() and (PROJECT_ROOT.parent / 'training_utils.py').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'training_utils.py').is_file():
    raise FileNotFoundError('Run from the loyalties repository or its notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_dataset import contextual_answer_token_ids, exact_answers_match
from training_utils import (
    plain_answer_choices, plain_correct_target, prepare_plain_sft_record,
    render_plain_prompt, wilson_interval,
)

# ---- User parameters: edit paths only here --------------------------------
DATA_DIR = Path(os.getenv('BIO_DATA_DIR', str(PROJECT_ROOT / 'data'))).expanduser().resolve()
RUN_DIR = None  # None = newest completed bio-knowledge run
SELECTED_CHECKPOINT = 'epoch_003_step_000960'  # folder name, 'final', or absolute adapter path

QWEN_REVISION = os.getenv('BASE_MODEL_REVISION', 'main')
TXGEMMA_9B_MODEL_ID = 'google/txgemma-9b-chat'
TXGEMMA_9B_REVISION = 'main'
TXGEMMA_27B_MODEL_ID = 'google/txgemma-27b-chat'
TXGEMMA_27B_REVISION = 'main'
MODEL_CACHE_DIR = os.getenv('MODEL_CACHE_DIR', str(PROJECT_ROOT / '.model_cache'))

RUN_MODELS = False
LOAD_IN_4BIT = True
BATCH_SIZE = 4
MAX_NEW_TOKENS = 64
MAX_PRIMARY_ITEMS = None     # None = full base-selection dev artifact
INCLUDE_LABBENCH_DEV_SLICE = True
LABBENCH_SLICE_COUNT = 100
SLICE_SEED = 1729

if BATCH_SIZE < 1 or MAX_NEW_TOKENS < 1 or LABBENCH_SLICE_COUNT < 1:
    raise ValueError('Batch size, generation limit, and slice size must be positive.')
display({'project_root': str(PROJECT_ROOT), 'data_dir': str(DATA_DIR), 'run_models': RUN_MODELS})


In [ ]:
if RUN_MODELS:
    from huggingface_hub import login, notebook_login
    hf_token = os.getenv('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print('Authenticated with HF_TOKEN.')
    else:
        print('HF_TOKEN is unset; opening Hugging Face login.')
        notebook_login()
else:
    print('Authentication skipped while RUN_MODELS=False.')


## 1. Resolve the selected adapter and model registry


In [ ]:
def newest_completed_run(root: Path) -> Path:
    candidates = [
        path for path in root.glob('bio-knowledge-*')
        if (path / 'adapter_bio_knowledge/adapter_config.json').is_file()
    ]
    if not candidates:
        raise FileNotFoundError(f'No completed biological-SFT run under {root}')
    return max(candidates, key=lambda path: path.stat().st_mtime)

RUNS_ROOT = PROJECT_ROOT / 'outputs/bio_knowledge_runs'
RUN_DIR = Path(RUN_DIR).expanduser().resolve() if RUN_DIR is not None else newest_completed_run(RUNS_ROOT).resolve()
ADAPTER_DIR = RUN_DIR / 'adapter_bio_knowledge'

def resolve_checkpoint(selection: str | Path) -> Path:
    candidate = Path(selection)
    if candidate.is_absolute():
        path = candidate
    elif str(selection) == 'final':
        path = ADAPTER_DIR
    else:
        path = ADAPTER_DIR / 'checkpoints' / str(selection)
    path = path.expanduser().resolve()
    if not (path / 'adapter_config.json').is_file():
        raise FileNotFoundError(path / 'adapter_config.json')
    return path

SELECTED_CHECKPOINT_PATH = resolve_checkpoint(SELECTED_CHECKPOINT)
adapter_config = __import__('json').loads((SELECTED_CHECKPOINT_PATH / 'adapter_config.json').read_text(encoding='utf-8'))
QWEN_MODEL_ID = adapter_config['base_model_name_or_path']

@dataclass(frozen=True)
class ModelSpec:
    label: str
    model_id: str
    revision: str
    parameters_billion: float
    adapter_path: Path | None = None

MODEL_SPECS = (
    ModelSpec('Qwen base', QWEN_MODEL_ID, QWEN_REVISION, 14.0),
    ModelSpec('Qwen + bio SFT', QWEN_MODEL_ID, QWEN_REVISION, 14.0, SELECTED_CHECKPOINT_PATH),
    ModelSpec('TxGemma 9B chat', TXGEMMA_9B_MODEL_ID, TXGEMMA_9B_REVISION, 9.0),
    ModelSpec('TxGemma 27B chat', TXGEMMA_27B_MODEL_ID, TXGEMMA_27B_REVISION, 27.0),
)
display(pd.DataFrame([{**asdict(spec), 'adapter_path': str(spec.adapter_path) if spec.adapter_path else None} for spec in MODEL_SPECS]))


## 2. Canonical development data

`DATA_DIR` is the only dataset root. The primary file is loaded either directly beneath it or from `DATA_DIR/splits`. Password/decoy copies are converted to the password-free SFT contract and deduplicated by `pair_id`, so every underlying problem is scored once.


In [ ]:
def find_data_file(filename: str, *, required: bool = True) -> Path | None:
    candidates = [DATA_DIR / filename, DATA_DIR / 'splits' / filename]
    for path in candidates:
        if path.is_file():
            return path
    if required:
        searched = '\n'.join(f'- {path}' for path in candidates)
        raise FileNotFoundError(f'Could not find {filename}. Searched:\n{searched}')
    return None

def read_jsonl(path: Path) -> list[dict]:
    import json
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def unique_plain_records(raw_rows: list[dict], *, allowed_sources: set[str] | None = None) -> list[dict]:
    unique = {}
    for raw in raw_rows:
        if raw.get('task_type') in {'nonbio', 'heldout_soft'}:
            continue
        source = str(raw.get('meta', {}).get('source', ''))
        if allowed_sources is not None and source not in allowed_sources:
            continue
        record = prepare_plain_sft_record(raw)
        identity = record['pair_id']
        signature = (render_plain_prompt(record), plain_correct_target(record), record['grading'])
        if identity in unique:
            if unique[identity][1] != signature:
                raise ValueError(f'Conflicting duplicate identity: {identity}')
            continue
        unique[identity] = (record, signature)
    return [value[0] for value in unique.values()]

primary_path = find_data_file('base_selection.jsonl')
primary_records = unique_plain_records(read_jsonl(primary_path))
if MAX_PRIMARY_ITEMS is not None:
    primary_records = primary_records[:MAX_PRIMARY_ITEMS]
if not primary_records:
    raise RuntimeError('The primary comparison set is empty.')

coverage = pd.DataFrame([{
    'task': row['meta'].get('gen_fn', 'unknown'),
    'difficulty': row['meta'].get('difficulty', 'unknown'),
    'grading': row['grading'],
} for row in primary_records])
display({'primary_path': str(primary_path), 'unique_items': len(primary_records)})
display(coverage.groupby(['task', 'difficulty', 'grading']).size().rename('n').reset_index())
for number, record in enumerate(primary_records[:2], 1):
    print(f"\n--- Primary example {number} ---")
    print(render_plain_prompt(record))
    print('Gold:', plain_correct_target(record))


### Independent LAB-Bench dev slice

This secondary slice is displayed separately and never blended into primary accuracy. Sampling is a deterministic hash ordering by `pair_id`.


In [ ]:
def deterministic_slice(records: list[dict], count: int, seed: int) -> list[dict]:
    import hashlib
    return sorted(records, key=lambda row: hashlib.sha256(f"{seed}:{row['pair_id']}".encode()).digest())[:count]

labbench_records = []
dev_path = find_data_file('dev.jsonl', required=False)
if INCLUDE_LABBENCH_DEV_SLICE and dev_path is not None:
    all_labbench = unique_plain_records(read_jsonl(dev_path), allowed_sources={'labbench'})
    labbench_records = deterministic_slice(all_labbench, LABBENCH_SLICE_COUNT, SLICE_SEED)
    display({'dev_path': str(dev_path), 'available_labbench': len(all_labbench), 'slice_n': len(labbench_records), 'seed': SLICE_SEED})
elif INCLUDE_LABBENCH_DEV_SLICE:
    print('dev.jsonl was not found under DATA_DIR; the optional LAB-Bench slice will be skipped.')
else:
    print('LAB-Bench dev slice disabled.')


## 3. Sequential model loading and deterministic scoring

All variants receive the exact same plain `Question … Answer:` serialization—no chat wrapper, system prompt, password, or key. Exact-match items use greedy generation; multiple-choice items use contextual next-token logits over their declared answer tokens. The selected LoRA is loaded on the same base model ID and revision as the untouched Qwen comparison.


In [ ]:
def resolve_revision(model_id: str, requested_revision: str) -> str:
    from huggingface_hub import HfApi
    return str(HfApi().model_info(model_id, revision=requested_revision).sha)

def load_model(spec: ModelSpec):
    import torch
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    if not torch.cuda.is_available():
        raise RuntimeError('A CUDA GPU is required for this four-model comparison.')
    resolved_revision = resolve_revision(spec.model_id, spec.revision)
    tokenizer_source = spec.adapter_path if spec.adapter_path and (spec.adapter_path / 'tokenizer_config.json').is_file() else spec.model_id
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source, revision=None if spec.adapter_path else resolved_revision,
        cache_dir=MODEL_CACHE_DIR, use_fast=True, trust_remote_code=False,
    )
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise RuntimeError(f'{spec.label}: tokenizer has neither pad nor EOS token')
        tokenizer.pad_token = tokenizer.eos_token
    architecture = AutoConfig.from_pretrained(spec.model_id, revision=resolved_revision, cache_dir=MODEL_CACHE_DIR)
    model_class = AutoModelForCausalLM
    if getattr(architecture, 'model_type', '') == 'qwen3_5':
        from transformers import AutoModelForMultimodalLM
        model_class = AutoModelForMultimodalLM
    kwargs = {
        'revision': resolved_revision, 'cache_dir': MODEL_CACHE_DIR,
        'device_map': 'auto', 'low_cpu_mem_usage': True,
        'trust_remote_code': False,
    }
    if LOAD_IN_4BIT:
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
        )
        kwargs['torch_dtype'] = torch.bfloat16
    else:
        kwargs['torch_dtype'] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    base = model_class.from_pretrained(spec.model_id, **kwargs)
    model = base
    if spec.adapter_path is not None:
        from peft import PeftModel
        model = PeftModel.from_pretrained(base, spec.adapter_path, is_trainable=False)
    model.eval()
    model.config.use_cache = True
    return model, tokenizer, resolved_revision

def model_input_device(model):
    return model.get_input_embeddings().weight.device

def release_model(model, tokenizer):
    import torch
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
def score_exact_batch(model, tokenizer, batch: list[dict], label: str, dataset: str) -> list[dict]:
    import torch
    prompts = [render_plain_prompt(record) for record in batch]
    encoded = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    input_width = encoded['input_ids'].shape[1]
    encoded = {key: value.to(model_input_device(model)) for key, value in encoded.items()}
    with torch.inference_mode():
        output = model.generate(
            **encoded, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    predictions = tokenizer.batch_decode(output[:, input_width:], skip_special_tokens=True)
    rows = []
    for record, prediction in zip(batch, predictions):
        prediction = prediction.strip()
        gold = plain_correct_target(record)
        rows.append({
            'variant': label, 'dataset': dataset, 'pair_id': record['pair_id'],
            'task': record['meta'].get('gen_fn') or record['meta'].get('subtask', 'unknown'),
            'difficulty': record['meta'].get('difficulty', 'unknown'),
            'source': record['meta']['source'], 'grading': record['grading'],
            'predicted': prediction, 'gold': gold,
            'is_correct': bool(exact_answers_match(prediction, gold, record)),
        })
    return rows

def score_choice_batch(model, tokenizer, batch: list[dict], label: str, dataset: str) -> list[dict]:
    import torch
    rows = []
    for record in batch:
        prompt = render_plain_prompt(record)
        encoded = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)
        encoded = {key: value.to(model_input_device(model)) for key, value in encoded.items()}
        choice_map = contextual_answer_token_ids(tokenizer, prompt, record)
        choices = plain_answer_choices(record)
        choice_ids = [choice_map[choice] for choice in choices]
        with torch.inference_mode():
            logits = model(**encoded).logits[0, -1, choice_ids].float()
        predicted = choices[int(torch.argmax(logits).item())]
        gold = plain_correct_target(record)
        rows.append({
            'variant': label, 'dataset': dataset, 'pair_id': record['pair_id'],
            'task': record['meta'].get('gen_fn') or record['meta'].get('subtask', 'unknown'),
            'difficulty': record['meta'].get('difficulty', 'unknown'),
            'source': record['meta']['source'], 'grading': record['grading'],
            'predicted': predicted, 'gold': gold, 'is_correct': predicted == gold,
        })
    return rows

def evaluate_records(model, tokenizer, records: list[dict], label: str, dataset: str) -> list[dict]:
    results = []
    for start in range(0, len(records), BATCH_SIZE):
        batch = records[start:start + BATCH_SIZE]
        modes = {record['grading'] for record in batch}
        if modes == {'exact_match'}:
            results.extend(score_exact_batch(model, tokenizer, batch, label, dataset))
        elif modes == {'choice_match'}:
            results.extend(score_choice_batch(model, tokenizer, batch, label, dataset))
        else:
            for record in batch:
                scorer = score_exact_batch if record['grading'] == 'exact_match' else score_choice_batch
                results.extend(scorer(model, tokenizer, [record], label, dataset))
        completed = min(start + len(batch), len(records))
        if completed % 100 == 0 or completed == len(records):
            print(f'{label} / {dataset}: {completed}/{len(records)}')
    return results


In [ ]:
predictions = []
resolved_revisions = {}
if RUN_MODELS:
    for number, spec in enumerate(MODEL_SPECS, 1):
        print(f"\n[{number}/{len(MODEL_SPECS)}] Loading {spec.label}: {spec.model_id}")
        model, tokenizer, resolved_revision = load_model(spec)
        resolved_revisions[spec.label] = resolved_revision
        predictions.extend(evaluate_records(model, tokenizer, primary_records, spec.label, 'base_selection_dev'))
        if labbench_records:
            predictions.extend(evaluate_records(model, tokenizer, labbench_records, spec.label, 'labbench_dev_slice'))
        release_model(model, tokenizer)
        print(f'Released {spec.label}.')
else:
    print('Dry run complete. Set RUN_MODELS=True to perform the sequential comparison.')


## 4. Per-model metrics

The primary table mirrors the pilot's task × difficulty report. Wilson 95% intervals are included, and LAB-Bench remains a separate secondary table.


In [ ]:
def metric_row(values: pd.DataFrame) -> dict:
    correct = int(values['is_correct'].sum())
    total = len(values)
    low, high = wilson_interval(correct, total)
    return {'correct': correct, 'n': total, 'accuracy': correct / total, 'wilson_low': low, 'wilson_high': high}

metrics = pd.DataFrame()
if predictions:
    predictions_df = pd.DataFrame(predictions)
    metric_rows = []
    grouping_specs = [
        ('overall', []), ('task', ['task']),
        ('task_difficulty', ['task', 'difficulty']), ('grading', ['grading']),
    ]
    for (variant, dataset), model_values in predictions_df.groupby(['variant', 'dataset'], sort=False):
        for grouping, columns in grouping_specs:
            if not columns:
                metric_rows.append({'variant': variant, 'dataset': dataset, 'grouping': grouping, 'group': 'all', **metric_row(model_values)})
                continue
            grouper = columns[0] if len(columns) == 1 else columns
            for keys, values in model_values.groupby(grouper):
                keys = (keys,) if not isinstance(keys, tuple) else keys
                metric_rows.append({'variant': variant, 'dataset': dataset, 'grouping': grouping, 'group': ' | '.join(map(str, keys)), **metric_row(values)})
    metrics = pd.DataFrame(metric_rows)
    display(Markdown('### Primary aggregate'))
    display(metrics.query("dataset == 'base_selection_dev' and grouping == 'overall'").round(4))
    display(Markdown('### Primary task × difficulty'))
    display(metrics.query("dataset == 'base_selection_dev' and grouping == 'task_difficulty'").round(4))
    if labbench_records:
        display(Markdown('### Secondary LAB-Bench dev slice'))
        display(metrics.query("dataset == 'labbench_dev_slice' and grouping in ['overall', 'task']").round(4))
    display(Markdown('### Exact resolved model revisions'))
    display(pd.DataFrame([{'variant': key, 'resolved_revision': value} for key, value in resolved_revisions.items()]))
else:
    print('Metrics will appear after RUN_MODELS=True completes.')


## 5. Visual and paired comparisons

Because every model scores the same identities, the paired table reports discordant wins and losses in addition to raw accuracy deltas. Its two-sided exact sign-test p-value is descriptive; it is not a new checkpoint-selection rule.


In [ ]:
def exact_sign_test_two_sided(wins: int, losses: int) -> float:
    n = wins + losses
    if n == 0:
        return 1.0
    k = min(wins, losses)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n)
    return min(1.0, 2 * tail)

paired_comparison = pd.DataFrame()
if predictions:
    primary = predictions_df.query("dataset == 'base_selection_dev'")
    correctness = primary.pivot(index='pair_id', columns='variant', values='is_correct').astype(bool)
    base_label = 'Qwen base'
    comparison_rows = []
    for label in [spec.label for spec in MODEL_SPECS if spec.label != base_label]:
        shared = correctness[[base_label, label]].dropna()
        wins = int((~shared[base_label] & shared[label]).sum())
        losses = int((shared[base_label] & ~shared[label]).sum())
        comparison_rows.append({
            'comparison': f'{label} − {base_label}', 'n': len(shared),
            'base_accuracy': shared[base_label].mean(), 'comparison_accuracy': shared[label].mean(),
            'accuracy_delta': shared[label].mean() - shared[base_label].mean(),
            'comparison_only_correct': wins, 'base_only_correct': losses,
            'exact_sign_test_p': exact_sign_test_two_sided(wins, losses),
        })
    paired_comparison = pd.DataFrame(comparison_rows)
    display(paired_comparison.round(4))

    aggregate = metrics.query("dataset == 'base_selection_dev' and grouping == 'overall'").copy()
    order = [spec.label for spec in MODEL_SPECS]
    aggregate['variant'] = pd.Categorical(aggregate['variant'], categories=order, ordered=True)
    aggregate = aggregate.sort_values('variant')
    yerr = np.vstack([aggregate['accuracy'] - aggregate['wilson_low'], aggregate['wilson_high'] - aggregate['accuracy']])
    fig, axis = plt.subplots(figsize=(8, 4.5))
    axis.bar(aggregate['variant'].astype(str), aggregate['accuracy'], yerr=yerr, capsize=4)
    axis.set(title='Primary canonical-dev comparison', ylabel='accuracy', ylim=(0, 1))
    axis.grid(axis='y', alpha=0.2)
    plt.xticks(rotation=15, ha='right')
    fig.tight_layout()
    plt.show()

    task_metrics = metrics.query("dataset == 'base_selection_dev' and grouping == 'task'")
    pivot = task_metrics.pivot(index='group', columns='variant', values='accuracy').reindex(columns=order)
    pivot.plot(kind='bar', figsize=(11, 5), ylim=(0, 1), ylabel='accuracy', title='Primary accuracy by task')
    plt.xticks(rotation=25, ha='right')
    plt.grid(axis='y', alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print('Comparisons and plots will appear after model execution.')


## Interpretation checklist

- Use the primary generated-dev results for the direct capability comparison.
- Treat LAB-Bench as a separate external/dev diagnostic; do not average it into the primary result.
- Compare the SFT checkpoint to **its own Qwen base** to measure the training gain.
- Compare TxGemma as an external therapeutic-domain reference, not as evidence that the Qwen training recipe succeeded or failed.
- Do not choose or retune the checkpoint from held-out test performance.
